In [ ]:
### Imports

# Auto-reload package modules whenever they change on disk, so edits to
# the usv_playpen source are picked up without restarting the kernel.
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
from pathlib import Path

from usv_playpen.analyses.npx_histology_stack_lightsheet_volume import stack_lightsheet_volume
from usv_playpen.analyses.npx_histology_stitch_smartspim_tiles import stitch_smartspim_tiles

# NPX Histology — TIFF Manipulations

Combine raw light-sheet microscopy acquisitions into single BigTIFF volumes for downstream histology / atlas-registration workflows. Two acquisition modalities are supported:

* **LaVision UltraMicroscope** (1.1× objective, 5.91 µm/pix lateral, 10 µm axial): a flat directory of OME-TIFF Z-planes per channel, single tile. Use `stack_lightsheet_volume` to glue the planes into one BigTIFF per channel. Optionally flips each plane in-plane (`xy_flip` ∈ `{'none', 'vertical', 'horizontal', 'both'}`; `'both'` ≡ 180° rotation) and reverses Z order for `vd` acquisitions so the output is dorsal-first.

* **LifeCanvas SmartSPIM** (1.625× objective, 4.02 µm/pix lateral, 10 µm axial): a tiled XY grid of Z-stacks per channel under `Ex_{wavelength}_Ch{n}/{X}/{X}_{Y}/`. Use `stitch_smartspim_tiles` to read `metadata.txt`, convert stage coordinates (0.1 µm units) to pixel offsets, and stream a plane-by-plane stitch with a bevel-shaped linear feather (`feather_pixels`) over tile seams.

**Channels.** Both functions default to `wavelength_nm=(488, 561)` and process both channels in one call, writing one BigTIFF per channel. To restrict to a single channel, pass an `int` (e.g. `wavelength_nm=561`).

**Per-call inputs (set inline in each cell below):**
* `raw_dir` — acquisition root.
* `output_path` — destination BigTIFF path. When multiple wavelengths are requested, this **must** contain the substring `{wavelength_nm}`, which is formatted per channel (e.g. `'.../{wavelength_nm}nm.tif'` → `'.../488nm.tif'` and `'.../561nm.tif'`).
* `wavelength_nm` — defaults to both channels; pass an int to restrict.

**Stable tunables** — `xy_flip`, `z_flip`, `feather_pixels`, `skip_first` — are loaded from `_parameter_settings/analyses_settings.json`. Override any of them by passing an explicit kwarg to the function call.

**Orientation controls.** Two independent knobs, both set by trial-and-error per dataset:

* `xy_flip` flips each 2D plane *in-plane* (one of `'none'`, `'vertical'` for top↔bottom, `'horizontal'` for left↔right, `'both'` = 180° rotation). It affects the **axial** view in napari.
* `z_flip` reverses the Z (depth) iteration order before writing. It affects the **coronal** and **sagittal** views in napari (both views share the Z axis, so a wrong-handed Z manifests in both at once).

We dropped the earlier auto-detection of acquisition direction (`dv` / `vd`) because the labels written by ImSpector / the SmartSPIM acquisition app proved unreliable across our datasets — every dataset still needed manual tweaking. Picking `z_flip` per dataset is simpler and explicit.

**Troubleshooting in napari.** If the brain renders flipped:
* Wrong in **axial** view → tweak `xy_flip`.
* Right in axial, **upside-down in coronal and sagittal** → toggle `z_flip`.

In [ ]:
### Stack a LaVision acquisition into BigTIFF volumes (one per channel)

with open(Path.cwd().parent / "_parameter_settings" / "analyses_settings.json", 'r') as analyses_settings_file:
    cfg = json.load(analyses_settings_file)['npx_histology_stack_lightsheet_volume']

raw_dir = '/mnt/lightsheet/_rawData/LaVision/bmimica/251015_bmimica_178621-dv-lv-1_09-44-40'
output_path = '/mnt/falkner/Bartul/histology/178621_2/registration/{wavelength_nm}nm/178621_{wavelength_nm}nm_fullsize.tif'

stack_lightsheet_volume(
    raw_dir=raw_dir,
    output_path=output_path,
    **cfg,
)

In [ ]:
### Stitch a SmartSPIM acquisition into BigTIFF volumes (one per channel)

with open(Path.cwd().parent / "_parameter_settings" / "analyses_settings.json", 'r') as analyses_settings_file:
    cfg = json.load(analyses_settings_file)['npx_histology_stitch_smartspim_tiles']

raw_dir = '/mnt/lightsheet/_rawData/SmartSPIM/jj9483/20251118_14_09_42_jj9483_181321_1x_vd_ss_1'
output_path = '/mnt/falkner/Bartul/histology/181321_1/registration/{wavelength_nm}nm/181321_{wavelength_nm}nm_stitched.tif'

stitch_smartspim_tiles(
    raw_dir=raw_dir,
    output_path=output_path,
    **cfg,
)